<a href="https://colab.research.google.com/github/penajuanmanuel6-hub/automated-etl-pipeline/blob/main/R%C3%ADoNilo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
import numpy as np
import pandas as pd

# ==========================================
# 1. SIMULACIÓN DE DATOS (La "Raíz")
# ==========================================
print("📥 Generando base de datos de sensores y campo del Delta del Nilo...")
np.random.seed(42)

n_records = 15000
governorates = ["Beheira", "Kafr El Sheikh", "Dakahlia", "Damietta"]

data = {
    "record_id": range(1, n_records + 1),
    "governorate": np.random.choice(governorates, n_records, p=[0.3, 0.3, 0.25, 0.15]),
    "soil_salinity_ec": np.random.normal(loc=4.2, scale=1.2, size=n_records), # Conductividad eléctrica (dS/m)
    "water_flow_m3s": np.random.normal(loc=120, scale=30, size=n_records),   # Caudal del río (m3/s)
    "crop_loss_pct": np.random.uniform(5, 50, n_records),                    # % de pérdida de cultivo
    "budget_allocated": np.random.choice([50000, 100000, 150000], n_records)
}

df_raw = pd.DataFrame(data)

# Introducir el incremento crítico del 38% en salinidad en zonas específicas
df_raw.loc[df_raw['governorate'].isin(['Kafr El Sheikh', 'Damietta']), 'soil_salinity_ec'] *= 1.38

# ==========================================
# 2. CARGA EN BASE DE DATOS Y CONSULTA SQL (El "Tronco")
# ==========================================
print("🪵 Cargando datos en SQLite y ejecutando consultas analíticas...")

# Crear base de datos en memoria
conn = sqlite3.connect(':memory:')
df_raw.to_sql('nile_climate_data', conn, index=False, if_exists='replace')

# Consulta SQL para identificar gobernaciones en alerta roja (agregación y filtrado)
query_sql = """
    SELECT
        governorate,
        COUNT(record_id) as total_reportes,
        ROUND(AVG(soil_salinity_ec), 2) as prom_salinidad_ec,
        ROUND(AVG(water_flow_m3s), 2) as prom_caudal_agua,
        ROUND(AVG(crop_loss_pct), 1) as prom_perdida_cultivo_pct,
        SUM(budget_allocated) as presupuesto_actual_zona
    FROM nile_climate_data
    GROUP BY governorate
    ORDER BY prom_salinidad_ec DESC;
"""

df_analisis = pd.read_sql(query_sql, conn)
print("\n--- RESULTADOS DEL ANÁLISIS TÉCNICO (SQL) ---")
print(df_analisis.to_string(index=False))

# ==========================================
# 3. TOMA DE DECISIONES (Las Ramas del Árbol)
# ==========================================
print("\n" + "="*50)
print("🌿 APLICACIÓN EN LAS TRAMAS DEL PNUD:")
print("="*50)

# Rama 1: Director de Orquesta (Reasignación operativa)
zona_critica = df_analisis.iloc[0]['governorate']
max_salinidad = df_analisis.iloc[0]['prom_salinidad_ec']

print(f"\n[RAMA 1 - Director de Orquesta]:")
print(f"👉 Alerta detectada en '{zona_critica}' con salinidad crítica de {max_salinidad} dS/m.")
print("Acción operativa: Se emite orden para reasignar el 40% del presupuesto estancado en maquinaria")
print(f"hacia la compra inmediata de sistemas de desalinización y riego por goteo en {zona_critica}.")

# Rama 2: El Traductor (Insumo para Política Pública)
print(f"\n[RAMA 2 - El Traductor (Policy Brief)]:")
print(f"📄 Resumen para el Ministerio de Medio Ambiente:")
print(f"   'Basado en el procesamiento de 15,000 registros, la gobernación de {zona_critica}")
print(f"   presenta una pérdida promedio de cultivos del {df_analisis.iloc[0]['prom_perdida_cultivo_pct']}%.")
print(f"   Se requiere declarar alerta fitosanitaria y de emergencia hídrica inmediata.'")

# Rama 3: El Arquitecto (Movilización de Recursos / GCF)
print(f"\n[RAMA 3 - El Arquitecto de Proyectos]:")
print(f"🏗️ Estructuración de Propuesta Internacional:")
print(f"   'El impacto sistémico supera el presupuesto actual de la oficina ($1.2M).")
print(f"   Se utiliza este modelo analítico para redactar la nota conceptual dirigida al")
print(f"   Fondo Verde del Clima (GCF) solicitando un fondo de $15M para resiliencia del Delta.'")

conn.close()

📥 Generando base de datos de sensores y campo del Delta del Nilo...
🪵 Cargando datos en SQLite y ejecutando consultas analíticas...

--- RESULTADOS DEL ANÁLISIS TÉCNICO (SQL) ---
   governorate  total_reportes  prom_salinidad_ec  prom_caudal_agua  prom_perdida_cultivo_pct  presupuesto_actual_zona
Kafr El Sheikh            4556               5.84            119.87                      27.3                454150000
      Damietta            2241               5.84            119.20                      27.6                222400000
       Beheira            4556               4.22            120.28                      27.7                452750000
      Dakahlia            3647               4.15            119.45                      27.4                366000000

🌿 APLICACIÓN EN LAS TRAMAS DEL PNUD:

[RAMA 1 - Director de Orquesta]:
👉 Alerta detectada en 'Kafr El Sheikh' con salinidad crítica de 5.84 dS/m.
Acción operativa: Se emite orden para reasignar el 40% del presupuesto estancad